# PRIMA articulation, stage by stage

This notebook runs the repository's production PRIMA Stage 1-to-SMAL components
on the same `SMAL_r` dog as the ANIMER visualization. It exposes detector
candidates, highest-IoU per-view selection, and the complete shared SMAL refinement
lifecycle.

Run it with `pkg/PRIMA/.venv/bin/python`. The expensive cells require a CUDA GPU,
the Stage 1 checkpoint under `pkg/PRIMA/data/PRIMAS1/`, BioCLIP assets, and
Detectron2 detector weights. The setup cell locates the repository root.

## 1. Setup and production components

In [ ]:
import os

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import sys
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "articulate").is_dir():
            return candidate
    local_fallback = Path("/home/knpob/work/code/atm-dev")
    if (local_fallback / "pyproject.toml").is_file():
        return local_fallback
    raise RuntimeError("Could not find the atm-dev repository root.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import torch
import trimesh
from tqdm.auto import tqdm

from articulate.shape_estimator.prima_estimator import (
    PRIMAShapeEstimator,
    select_best_aligned_candidate,
)
from articulate.shape_loader.animer_mesh_loader import AniMerMeshShapeLoader
from articulate.shape_prior.prima_smal_prior import PRIMASMALShapePrior
from articulate.shape_refiner.prima_smal_refiner import PRIMASMALShapeRefiner
from articulate.shape_refiner.smal_refiner import chamfer_distance, crop_keypoints_to_image
from articulate.view_sampler.multi_view_sampler import MultiViewSampler

PYVISTA_BACKEND = "static"
EXAMPLE_PATH = Path("/home/knpob/work/data/shape-corr/SMAL_r/off/dog_01.off")
DEVICE = "cuda"
SEED = 0
N_ITERS_KEYPOINTS = 1000
N_ITERS_SURFACE = 1000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pv.set_jupyter_backend(PYVISTA_BACKEND)
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

PRIMA_ROOT = PROJECT_ROOT / "pkg/PRIMA"
CHECKPOINT_PATH = PRIMA_ROOT / "data/PRIMAS1/checkpoints/s1ckpt_inference.ckpt"
CONFIG_PATH = PRIMA_ROOT / "data/PRIMAS1/.hydra/config.yaml"
SMAL_ASSETS = PRIMA_ROOT / "data/smal"

required_paths = [EXAMPLE_PATH, PRIMA_ROOT, CHECKPOINT_PATH, CONFIG_PATH, SMAL_ASSETS]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f"Missing required assets: {missing}"
assert DEVICE != "cuda" or torch.cuda.is_available(), "CUDA is required for the configured PRIMA run."

print(f"Repository: {PROJECT_ROOT}")
print(f"Shape:      {EXAMPLE_PATH}")
print(f"Device:     {DEVICE}")

In [ ]:
shape_loader = AniMerMeshShapeLoader(shape_dir=str(EXAMPLE_PATH.parent))
view_sampler = MultiViewSampler(
    elevation_config="multi_3level",
    rotation_step=45.0,
    camera_dist=4.0,
    axis_order="xyz",
    camera_configs={
        "focal_length": 1000.0,
        "camera_center": (256, 256),
        "h": 512,
        "w": 512,
    },
)
shape_estimator = PRIMAShapeEstimator(
    device=DEVICE,
    prima_pkg_path=str(PRIMA_ROOT),
    checkpoint_path=str(CHECKPOINT_PATH),
    detector_config="COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml",
    detector_weights="https://dl.fbaipublicfiles.com/detectron2/COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x/139173657/model_final_68b088.pkl",
    detector_score_threshold=0.5,
    animal_score_threshold=0.7,
    animal_class_ids=[15, 16, 17, 18, 19, 21, 22],
    duplicate_iou_threshold=0.5,
    containment_threshold=0.9,
    full_frame_fallback=True,
    iou_threshold=0.7,
    retry_iou_threshold=True,
    iou_retry_step=0.1,
    min_iou_threshold=0.0,
    ransac_n_iters=3000,
    ransac_sample_size=8,
    ransac_inlier_thresh=0.02,
    ransac_min_inliers=500,
    ransac_seed=SEED,
)
shape_prior = PRIMASMALShapePrior(device=DEVICE)
shape_refiner = PRIMASMALShapeRefiner(
    device=DEVICE,
    sigma_px=5.0,
    conf_thresh=1e-2,
    triangulation_iters=2,
    optimize_keypoints=True,
    lr_keypoints=1e-3,
    n_iters_keypoints=N_ITERS_KEYPOINTS,
    optimize_surface=True,
    n_views_surface=3,
    merge_method="mean",
    lr_surface=1e-2,
    n_iters_surface=N_ITERS_SURFACE,
    bone_term_weight=0.0,
    keypoint_term_weight=0.0,
    src_to_tgt=True,
    src_subsample_num=5000,
    tgt_subsample_num=5000,
)

In [ ]:
def plot_image_grid(images, titles, *, columns=6, cmap=None, figsize_scale=3.0):
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns * figsize_scale, rows * figsize_scale))
    axes = np.atleast_1d(axes).ravel()
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    for ax in axes[len(images):]:
        ax.axis("off")
    plt.tight_layout()
    return fig

def add_skeleton(plotter, points, bone_pairs, *, color="teal", width=4):
    points = np.asarray(points)
    for i, j in bone_pairs:
        if np.isfinite(points[[i, j]]).all():
            plotter.add_lines(points[[i, j]], color=color, width=width)

def show_overlay(mesh_target, mesh_smal=None, *, points=None, skeletons=(), camera="xy", title=None):
    plotter = pv.Plotter()
    plotter.add_mesh(pv.wrap(mesh_target), color="white", opacity=0.35, label="target")
    if mesh_smal is not None:
        plotter.add_mesh(pv.wrap(mesh_smal), color="salmon", opacity=0.75, label="SMAL")
    if points is not None:
        plotter.add_points(np.asarray(points), color="navy", point_size=7, render_points_as_spheres=True)
    for skeleton_points, pairs, color in skeletons:
        add_skeleton(plotter, skeleton_points, pairs, color=color)
    plotter.camera_position = camera
    if title:
        plotter.add_title(title)
    plotter.add_legend()
    return plotter.show()

def as_smal_mesh(vertices, faces):
    if torch.is_tensor(vertices):
        vertices = vertices.detach().cpu().numpy()
    return trimesh.Trimesh(vertices=vertices, faces=faces, process=False)

def joint_rmse(predicted, target):
    predicted = np.asarray(predicted)
    target = np.asarray(target)
    valid = np.isfinite(predicted).all(axis=-1) & np.isfinite(target).all(axis=-1)
    return float(np.sqrt(np.mean((predicted[valid] - target[valid]) ** 2)))

def rgba_overlay(rgb, rgba):
    rgb = rgb.astype(np.float32) / 255.0
    rgba = rgba.astype(np.float32) / 255.0
    alpha = rgba[..., 3:4]
    rendered = rgba[..., :3] * alpha + rgb * (1.0 - alpha)
    return np.clip(0.55 * rgb + 0.45 * rendered, 0.0, 1.0)

## 2. Load one shape and render the production camera views

In [ ]:
shape = shape_loader.pipeline(str(EXAMPLE_PATH))
mesh = shape["mesh"]
print(f"{shape['name']}: {len(mesh.vertices):,} vertices, {len(mesh.faces):,} faces, area={mesh.area:.4f}")
show_overlay(mesh, camera="xy", title="Input shape after the shared animal coordinate transform")

In [ ]:
camera_views = view_sampler.sample_views()
view_sampling = view_sampler.pipeline(shape=shape, camera_views=camera_views)
observations = view_sampling["observations"]
assert len(observations) == len(camera_views) == 18, "At least one configured view failed to render."
print(f"Rendered {len(observations)} views: {list(observations)}")

In [ ]:
view_titles = [f"az={az:g}, el={el:g}" for az, el in observations]
plot_image_grid([view["rgb"] for view in observations.values()], view_titles)
plot_image_grid([view["depth"] for view in observations.values()], view_titles, cmap="viridis")

## 3. PRIMA inference, candidate selection, and similarity alignment

PRIMA may retain several detector boxes for one rendered view. Each box is inferred
and aligned once at an IoU threshold of zero. The highest-IoU aligned candidate is
retained per view, then the production retry threshold is applied to those winners.

In [ ]:
shape_estimator.init_estimator()
shape_prior.load()
shape_prior.load_from_estimator(shape_estimator)
print("PRIMA Stage 1 estimator and checkpoint-owned SMAL prior loaded.")

In [ ]:
prima_estimates = {}
inference_errors = {}
for view_key, view_data in tqdm(observations.items(), desc="PRIMA inference"):
    try:
        prima_estimates[view_key] = shape_estimator.estimate(
            img_rgb=view_data["rgb"],
            view_key=view_key,
        )
    except Exception as error:
        inference_errors[view_key] = repr(error)

successful_views = sum(bool(estimates) for estimates in prima_estimates.values())
total_candidates = sum(len(estimates) for estimates in prima_estimates.values())
print(f"Successful inference views: {successful_views}/{len(observations)}")
print(f"Retained PRIMA candidates:  {total_candidates}")
if inference_errors:
    print("Failed views:", inference_errors)
assert total_candidates, "PRIMA did not produce a candidate for any view."

In [ ]:
prediction_panels = []
prediction_titles = []
prediction_keypoints = []
for view_key, estimates in prima_estimates.items():
    for candidate_index, estimate in enumerate(estimates):
        prediction_panels.append(observations[view_key]["rgb"].copy())
        prediction_titles.append(f"{view_key}, candidate {candidate_index}")
        prediction_keypoints.append(
            crop_keypoints_to_image(
                estimate["prima_output"], estimate["box_center"], estimate["box_size"],
                view_sampling["camera_configs"]["h"],
            )
        )

fig = plot_image_grid(prediction_panels, prediction_titles, columns=4)
for ax, keypoints in zip(fig.axes, prediction_keypoints):
    ax.scatter(keypoints[:, 0], keypoints[:, 1], s=6, marker="x", c="cyan")
fig.suptitle("Retained PRIMA candidates with crop-to-image 2D keypoints", y=1.01)

In [ ]:
configured_iou_threshold = shape_estimator.iou_threshold
shape_estimator.iou_threshold = 0.0
aligned_candidates_by_view = {}
alignment_errors = {}
for view_key, estimates in tqdm(prima_estimates.items(), desc="Candidate similarity alignment"):
    aligned = []
    for candidate_index, estimate in enumerate(estimates):
        try:
            result = shape_estimator.se3_align(
                mesh=mesh,
                view_data=observations[view_key],
                estimate=estimate,
            )
            if result is not None:
                aligned.append(result)
        except Exception as error:
            alignment_errors[(view_key, candidate_index)] = repr(error)
    aligned_candidates_by_view[view_key] = aligned

best_alignment_candidates = {
    view_key: select_best_aligned_candidate(candidates)
    for view_key, candidates in aligned_candidates_by_view.items()
    if candidates
}
best_alignment_candidates = {
    key: value for key, value in best_alignment_candidates.items() if value is not None
}

current_iou = configured_iou_threshold
while current_iou >= shape_estimator.min_iou_threshold:
    results = {key: value for key, value in best_alignment_candidates.items() if value["iou"] >= current_iou}
    if results or not shape_estimator.retry_iou_threshold or current_iou == shape_estimator.min_iou_threshold:
        break
    current_iou = round(current_iou - shape_estimator.iou_retry_step, 1)

shape_estimator.iou_threshold = current_iou
print(f"Accepted {len(results)}/{len(best_alignment_candidates)} per-view winners at IoU >= {current_iou:.1f}")
if alignment_errors:
    print("Candidate alignment failures:", alignment_errors)
assert len(results) >= 2, "Fewer than two valid views remain; triangulation is not defined."

In [ ]:
candidate_rows = []
for view_key, candidates in aligned_candidates_by_view.items():
    winner = best_alignment_candidates.get(view_key)
    for candidate_index, candidate in enumerate(candidates):
        candidate_rows.append((view_key, candidate_index, candidate, candidate is winner))

labels = [f"{key[0]:g}/{key[1]:g} c{index}" for key, index, _, _ in candidate_rows]
ious = [candidate["iou"] for _, _, candidate, _ in candidate_rows]
colors = [
    "tab:green" if is_winner and key in results else "tab:red" if is_winner else "tab:gray"
    for key, _, _, is_winner in candidate_rows
]

fig, ax = plt.subplots(figsize=(max(14, len(candidate_rows) * 0.55), 4))
ax.bar(np.arange(len(candidate_rows)), ious, color=colors)
ax.axhline(current_iou, color="black", linestyle="--", label=f"threshold={current_iou:.1f}")
ax.set_xticks(np.arange(len(candidate_rows)), labels, rotation=60, ha="right")
ax.set_ylabel("silhouette IoU")
ax.set_xlabel("azimuth/elevation and candidate index")
ax.set_title("PRIMA candidate IoUs; green/red bars are per-view winners")
ax.legend()
plt.tight_layout()

In [ ]:
overlay_images = [rgba_overlay(observations[key]["rgb"], value["img_rgba_smal"]) for key, value in results.items()]
overlay_titles = [f"{key}, winner IoU={value['iou']:.3f}" for key, value in results.items()]
plot_image_grid(overlay_images, overlay_titles, columns=3)

In [ ]:
plotter = pv.Plotter()
plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.25, label="target")
for result in results.values():
    plotter.add_mesh(pv.wrap(result["mesh_smal_aligned"]), color="salmon", opacity=0.15)
plotter.camera_position = "xy"
plotter.add_title("All accepted per-view PRIMA SMAL alignments")
plotter.add_legend()
plotter.show()

## 4. Robust multi-view SMAL triangulation

In [ ]:
shape_estimation = {
    "shape": shape,
    "results": results,
    "camera_configs": view_sampling["camera_configs"],
    "axis_order": view_sampling["axis_order"],
    "estimator": shape_estimator,
}
triangulated, used_views, conf_by_view = shape_refiner.joint_triangulate(
    results, camera_configs=shape_estimation["camera_configs"]
)
finite_keypoints = np.isfinite(triangulated).all(axis=1)
print(f"Finite triangulated joints: {finite_keypoints.sum()}/{len(triangulated)}")
print(f"Views used per joint: min={used_views.min()}, median={np.median(used_views):g}, max={used_views.max()}")
assert len(triangulated) == 26, f"Expected 26 SMAL joints, got {len(triangulated)}"
assert finite_keypoints.all(), "Triangulation produced non-finite joints; inspect confidence and accepted views."

In [ ]:
smal_bones = shape_prior.get_skeleton()
show_overlay(
    mesh,
    points=triangulated,
    skeletons=[(triangulated, smal_bones, "teal")],
    camera="xy",
    title="Robustly triangulated 26-joint PRIMA/SMAL skeleton",
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7), constrained_layout=True)
axes[0].bar(np.arange(len(used_views)), used_views)
axes[0].set(title="Views retained per triangulated joint", xlabel="SMAL joint ID", ylabel="views")
im = axes[1].imshow(conf_by_view, aspect="auto", cmap="magma", vmin=0, vmax=1)
axes[1].set(title="Tukey confidence after reprojection gating", xlabel="SMAL joint ID", ylabel="accepted view index")
fig.colorbar(im, ax=axes[1], label="confidence")

## 5. Optimize SMAL pose against triangulated joints

In [ ]:
betas_initial, global_orient_initial, pose_initial = shape_refiner.stack_smal_params(results)
R_all, t_all, s_all = shape_refiner.stack_alignment_params(results, dtype=betas_initial.dtype)
vertices_initial, joints_initial = shape_prior.generate(
    betas_initial, global_orient_initial, pose_initial, R_all, s_all, t_all
)
initial_rmse = joint_rmse(
    joints_initial.detach().cpu().numpy(),
    np.broadcast_to(triangulated, joints_initial.shape),
)
print(f"Initial multi-view joint RMSE: {initial_rmse:.6f}")

In [ ]:
betas_pose, global_orient_pose, pose_pose = shape_refiner.pose_optim(
    results=results, triangulated=triangulated, shape_prior=shape_prior
)
vertices_pose, joints_pose = shape_prior.generate(
    betas_pose, global_orient_pose, pose_pose, R_all, s_all, t_all
)
pose_rmse = joint_rmse(
    joints_pose.detach().cpu().numpy(),
    np.broadcast_to(triangulated, joints_pose.shape),
)
print(f"Pose-optimized multi-view joint RMSE: {pose_rmse:.6f}")
print(f"RMSE reduction: {initial_rmse - pose_rmse:.6f}")

In [ ]:
view_index = 0
initial_mesh = as_smal_mesh(vertices_initial[view_index], shape_prior.faces)
pose_mesh = as_smal_mesh(vertices_pose[view_index], shape_prior.faces)

plotter = pv.Plotter(shape=(1, 2), window_size=(1400, 650))
for column, (stage_name, stage_mesh, stage_joints) in enumerate([
    ("Before joint optimization", initial_mesh, joints_initial[view_index].detach().cpu().numpy()),
    ("After joint optimization", pose_mesh, joints_pose[view_index].detach().cpu().numpy()),
]):
    plotter.subplot(0, column)
    plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.25)
    plotter.add_mesh(pv.wrap(stage_mesh), color="salmon", opacity=0.65)
    add_skeleton(plotter, triangulated, smal_bones, color="teal")
    add_skeleton(plotter, stage_joints, smal_bones, color="red")
    plotter.add_title(stage_name)
    plotter.camera_position = "xy"
plotter.show()

## 6. Select the best views for surface optimization

PRIMA uses the complete input surface, as does ANIMER; neither SMAL preset trims
the target before optimization.

In [ ]:
alignment_inliers = np.concatenate([result["alignment"]["points_tgt_ori"] for result in results.values()], axis=0)
surface_samples, _ = trimesh.sample.sample_surface_even(mesh, count=5000)

plotter = pv.Plotter(shape=(1, 2), window_size=(1300, 600))
plotter.subplot(0, 0)
plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.25)
plotter.add_points(alignment_inliers, color="navy", point_size=2)
plotter.add_title("Accepted winner alignment inliers")
plotter.camera_position = "xy"
plotter.subplot(0, 1)
plotter.add_points(surface_samples, scalars=surface_samples[:, 1], cmap="viridis", point_size=3)
plotter.add_title("5,000 sampled target points")
plotter.camera_position = "xy"
plotter.show()

In [ ]:
view_keys = list(results)
chamfer_by_view = {
    key: chamfer_distance(mesh.vertices, vertices_pose[index].detach().cpu().numpy())
    for index, key in enumerate(view_keys)
}
selected_keys = [
    key for _, key in sorted((score, key) for key, score in chamfer_by_view.items())[:shape_refiner.n_views_surface]
]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(
    np.arange(len(view_keys)),
    [chamfer_by_view[key] for key in view_keys],
    color=["tab:green" if key in selected_keys else "tab:gray" for key in view_keys],
)
ax.set_xticks(np.arange(len(view_keys)), [f"{az:g}/{el:g}" for az, el in view_keys], rotation=45, ha="right")
ax.set(title="Chamfer-based view selection after pose optimization", xlabel="azimuth/elevation", ylabel="bidirectional squared Chamfer")
plt.tight_layout()
print("Selected views:", selected_keys)

## 7. Surface optimization and final merge

The inherited production lifecycle selects the best views, restarts them from
their original PRIMA estimates, fits the target surface, and merges corresponding
SMAL vertices by their mean.

In [ ]:
betas_surface, global_orient_surface, pose_surface, results_for_surface = shape_refiner.shape_optim(
    results=results,
    mesh=mesh,
    triangulated=triangulated,
    shape_prior=shape_prior,
    betas=betas_pose,
    global_orient=global_orient_pose,
    pose=pose_pose,
)
assert list(results_for_surface) == selected_keys, "Production selection disagrees with the diagnostic ranking."
print("Surface optimization used:", list(results_for_surface))

In [ ]:
R_surface, t_surface, s_surface = shape_refiner.stack_alignment_params(
    results_for_surface, dtype=betas_surface.dtype
)
vertices_surface, joints_surface = shape_prior.generate(
    betas_surface, global_orient_surface, pose_surface,
    R_surface, s_surface, t_surface,
)
mesh_merged = shape_refiner.merge_meshes(
    results=results_for_surface,
    mesh_target=mesh,
    shape_prior=shape_prior,
    betas=betas_surface,
    global_orient=global_orient_surface,
    pose=pose_surface,
)

surface_chamfers = [
    chamfer_distance(mesh.vertices, vertices_surface[i].detach().cpu().numpy())
    for i in range(len(results_for_surface))
]
final_chamfer = chamfer_distance(mesh.vertices, mesh_merged.vertices)
surface_rmse = joint_rmse(
    joints_surface.detach().cpu().numpy(),
    np.broadcast_to(triangulated, joints_surface.shape),
)
print(f"Selected-view surface Chamfer: {[round(value, 6) for value in surface_chamfers]}")
print(f"Final merged Chamfer:          {final_chamfer:.6f}")
print(f"Final selected-view joint RMSE: {surface_rmse:.6f}")

In [ ]:
selected_view_index = view_keys.index(selected_keys[0])
stage_meshes = [
    ("Initial estimate", as_smal_mesh(vertices_initial[selected_view_index], shape_prior.faces)),
    ("Pose refined", as_smal_mesh(vertices_pose[selected_view_index], shape_prior.faces)),
    ("Surface refined", as_smal_mesh(vertices_surface[0], shape_prior.faces)),
    ("Final mean merge", mesh_merged),
]
plotter = pv.Plotter(shape=(1, 4), window_size=(2000, 520))
for column, (stage_name, stage_mesh) in enumerate(stage_meshes):
    plotter.subplot(0, column)
    plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.25)
    plotter.add_mesh(pv.wrap(stage_mesh), color="salmon", opacity=0.7)
    plotter.add_title(stage_name)
    plotter.camera_position = "xy"
plotter.show()

In [ ]:
plotter = pv.Plotter(shape=(1, 3), window_size=(1800, 600))
for column, camera in enumerate(["xy", "xz", "zy"]):
    plotter.subplot(0, column)
    plotter.add_mesh(pv.wrap(mesh), color="white", opacity=0.3, label="input")
    plotter.add_mesh(pv.wrap(mesh_merged), color="salmon", opacity=0.7, label="merged SMAL")
    plotter.add_title(f"Final overlay: {camera}")
    plotter.camera_position = camera
plotter.add_legend()
plotter.show()

In [ ]:
diagnostics = {
    "shape": shape["name"],
    "rendered_views": len(observations),
    "successful_inference_views": successful_views,
    "retained_detector_candidates": total_candidates,
    "aligned_candidate_count": sum(len(values) for values in aligned_candidates_by_view.values()),
    "accepted_alignment_views": len(results),
    "effective_iou_threshold": current_iou,
    "mean_accepted_iou": float(np.mean([result["iou"] for result in results.values()])),
    "finite_triangulated_joints": int(finite_keypoints.sum()),
    "initial_joint_rmse": initial_rmse,
    "pose_joint_rmse": pose_rmse,
    "surface_joint_rmse": surface_rmse,
    "selected_surface_views": list(results_for_surface),
    "final_merged_chamfer": final_chamfer,
}
diagnostics

## Visual correctness checklist

- The transformed animal should be upright and centered in the production views.
- Duplicate detections should be suppressed; multiple retained candidates should be compared by IoU.
- Green candidate bars should be the highest-IoU winners for accepted views.
- Crop-to-image keypoints should lie on the animal without a vertical flip.
- Accepted SMAL meshes and the triangulated skeleton should share the target coordinate frame.
- Joint optimization should reduce RMSE and move the red skeleton toward the teal skeleton.
- Selected surface views should have the lowest displayed Chamfer scores.
- Surface optimization and final mean merging should preserve topology and transform direction.